In [2]:
import pandas as pd

donors = pd.read_csv("HLA_donors.tsv", sep="\t")
hits = pd.read_csv("HLA_sample_hits.tsv", sep="\t")
peptides = pd.read_csv("HLA_peptides.tsv", sep="\t")

a0101_donors = donors.loc[donors["hla_allele"].str.contains(r"A\*?01:01"), "donor"].unique()

candidate_hits = hits[(hits["donor"].isin(a0101_donors)) & (hits["hla_class"].str.contains("I", case=False))]

candidates = peptides[peptides["peptide_sequence_id"].isin(candidate_hits["peptide_sequence_id"])]
candidates = candidates[candidates["peptide_sequence"].str.len() == 9]

In [3]:
len(candidates)

25340

In [4]:
protein_map = pd.read_csv("HLA_protein_map.tsv", sep="\t")

candidate_proteins = protein_map[protein_map["peptide_sequence_id"].isin(candidates["peptide_sequence_id"])]

n_survived = candidates["peptide_sequence_id"].nunique()
n_source_proteins = candidate_proteins["uniprot_id"].nunique()

print(f"{n_survived} peptides survived, from {n_source_proteins} distinct source proteins")

25340 peptides survived, from 9857 distinct source proteins


In [5]:
import pandas as pd

cardiac_tissues = ["Heart"]
neural_tissues = ["Brain", "Cerebellum", "Myelon"]

candidate_hits = hits[hits["peptide_sequence_id"].isin(candidates["peptide_sequence_id"])]
tissue_sets = candidate_hits.groupby("peptide_sequence_id")["tissue"].apply(set)

flags = tissue_sets.apply(lambda t: pd.Series({
    "cardiac": bool(t & set(cardiac_tissues)),
    "neural": bool(t & set(neural_tissues)),
    "tissues": sorted(t),
}))

flagged = candidates.merge(flags, left_on="peptide_sequence_id", right_index=True)

print(f"{flagged['cardiac'].sum()} peptides observed in cardiac tissue")
print(f"{flagged['neural'].sum()} peptides observed in neural tissue")

unmapped = flagged[~flagged["peptide_sequence_id"].isin(protein_map["peptide_sequence_id"])]
print(f"{len(unmapped)} candidate peptides have no source protein annotation")

merged = flagged.merge(protein_map, on="peptide_sequence_id")

source_summary = merged.groupby("uniprot_id").agg(
    n_peptides=("peptide_sequence_id", "nunique"),
    cardiac=("cardiac", "any"),
    neural=("neural", "any"),
).reset_index()

print(f"{source_summary.shape[0]} distinct source proteins")
print(f"{source_summary['cardiac'].sum()} sources flagged cardiac")
print(f"{source_summary['neural'].sum()} sources flagged neural")

cardiac_sources = source_summary[source_summary["cardiac"]]
neural_sources = source_summary[source_summary["neural"]]

3354 peptides observed in cardiac tissue
4884 peptides observed in neural tissue
0 candidate peptides have no source protein annotation
9857 distinct source proteins
2531 sources flagged cardiac
3542 sources flagged neural


In [6]:
candidates.to_csv("a0101_9mer_candidates.csv", index=False)

In [7]:
f"{source_summary['cardiac'].sum()} cardiac"

'2531 cardiac'

In [8]:
f"{source_summary['neural'].sum()} neural"

np.int64(3542)